In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("BigDataFraudDetection") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 4.2.0


In [2]:
df_spark = spark.read.csv(
    "../data/processed/creditcard_clean.csv",
    header=True,
    inferSchema=True
)

print("Rows:", df_spark.count())
print("Columns:", len(df_spark.columns))

Rows: 284807
Columns: 31


In [3]:
df_spark.show(5)

+----+------------------+-------------------+----------------+------------------+-------------------+-------------------+-------------------+------------------+------------------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+-------------------+-------------------+------------------+------------------+------------------+------------------+------------------+-------------------+-------------------+------+-----+
|Time|                V1|                 V2|              V3|                V4|                 V5|                 V6|                 V7|                V8|                V9|                V10|               V11|               V12|               V13|               V14|               V15|               V16|               V17|                V18|               V19|                V20|                V21|               V22|          

In [4]:
df_spark.printSchema()

root
 |-- Time: double (nullable = true)
 |-- V1: double (nullable = true)
 |-- V2: double (nullable = true)
 |-- V3: double (nullable = true)
 |-- V4: double (nullable = true)
 |-- V5: double (nullable = true)
 |-- V6: double (nullable = true)
 |-- V7: double (nullable = true)
 |-- V8: double (nullable = true)
 |-- V9: double (nullable = true)
 |-- V10: double (nullable = true)
 |-- V11: double (nullable = true)
 |-- V12: double (nullable = true)
 |-- V13: double (nullable = true)
 |-- V14: double (nullable = true)
 |-- V15: double (nullable = true)
 |-- V16: double (nullable = true)
 |-- V17: double (nullable = true)
 |-- V18: double (nullable = true)
 |-- V19: double (nullable = true)
 |-- V20: double (nullable = true)
 |-- V21: double (nullable = true)
 |-- V22: double (nullable = true)
 |-- V23: double (nullable = true)
 |-- V24: double (nullable = true)
 |-- V25: double (nullable = true)
 |-- V26: double (nullable = true)
 |-- V27: double (nullable = true)
 |-- V28: double (nulla

In [5]:
df_spark.groupBy("Class").count().show()

+-----+------+
|Class| count|
+-----+------+
|    1|   492|
|    0|284315|
+-----+------+



In [6]:
df_spark.printSchema()

root
 |-- Time: double (nullable = true)
 |-- V1: double (nullable = true)
 |-- V2: double (nullable = true)
 |-- V3: double (nullable = true)
 |-- V4: double (nullable = true)
 |-- V5: double (nullable = true)
 |-- V6: double (nullable = true)
 |-- V7: double (nullable = true)
 |-- V8: double (nullable = true)
 |-- V9: double (nullable = true)
 |-- V10: double (nullable = true)
 |-- V11: double (nullable = true)
 |-- V12: double (nullable = true)
 |-- V13: double (nullable = true)
 |-- V14: double (nullable = true)
 |-- V15: double (nullable = true)
 |-- V16: double (nullable = true)
 |-- V17: double (nullable = true)
 |-- V18: double (nullable = true)
 |-- V19: double (nullable = true)
 |-- V20: double (nullable = true)
 |-- V21: double (nullable = true)
 |-- V22: double (nullable = true)
 |-- V23: double (nullable = true)
 |-- V24: double (nullable = true)
 |-- V25: double (nullable = true)
 |-- V26: double (nullable = true)
 |-- V27: double (nullable = true)
 |-- V28: double (nulla

In [7]:
df_spark.groupBy("Class").count().show()

+-----+------+
|Class| count|
+-----+------+
|    1|   492|
|    0|284315|
+-----+------+



In [8]:
fraud_spark = df_spark.filter(df_spark["Class"] == 1)

print("Fraud transactions:", fraud_spark.count())

Fraud transactions: 492


In [9]:
fraud_spark.select(
    "Time",
    "Amount",
    "Class"
).show(10)

+------+------+-----+
|  Time|Amount|Class|
+------+------+-----+
| 406.0|   0.0|    1|
| 472.0| 529.0|    1|
|4462.0|239.93|    1|
|6986.0|  59.0|    1|
|7519.0|   1.0|    1|
|7526.0|   1.0|    1|
|7535.0|   1.0|    1|
|7543.0|   1.0|    1|
|7551.0|   1.0|    1|
|7610.0|   1.0|    1|
+------+------+-----+
only showing top 10 rows


In [10]:
print(df_spark.columns)

['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']


In [11]:
feature_columns = [col for col in df_spark.columns if col != "Class"]

print("Number of features:", len(feature_columns))
print(feature_columns)

Number of features: 30
['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']


In [12]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

df_features = assembler.transform(df_spark)

df_features.select("features", "Class").show(5, truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|features                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

In [13]:
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(
    inputCol="features",
    outputCol="scaledFeatures",
    withStd=True,
    withMean=True
)

scaler_model = scaler.fit(df_features)

df_scaled = scaler_model.transform(df_features)

df_scaled.select("scaledFeatures", "Class").show(5, truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|scaledFeatures                                                                                                                                                                                                                                                                                                                                                                                      

In [14]:
train_spark, test_spark = df_scaled.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Training rows:", train_spark.count())
print("Testing rows:", test_spark.count())

Training rows: 227790
Testing rows: 57017


In [15]:
print("Training distribution:")
train_spark.groupBy("Class").count().show()

print("Testing distribution:")
test_spark.groupBy("Class").count().show()

Training distribution:


+-----+------+
|Class| count|
+-----+------+
|    1|   396|
|    0|227394|
+-----+------+

Testing distribution:
+-----+-----+
|Class|count|
+-----+-----+
|    1|   96|
|    0|56921|
+-----+-----+



In [17]:
spark_summary = df_spark.select(
    "Time",
    "Amount"
).summary()

spark_summary.show()

+-------+-----------------+------------------+
|summary|             Time|            Amount|
+-------+-----------------+------------------+
|  count|           284807|            284807|
|   mean|94813.85957508067|  88.3496192509521|
| stddev|47488.14595456643|250.12010924018836|
|    min|              0.0|               0.0|
|    25%|          54190.0|              5.59|
|    50%|          84680.0|              22.0|
|    75%|         139320.0|             77.14|
|    max|         172792.0|          25691.16|
+-------+-----------------+------------------+



In [18]:
df_spark.selectExpr(
    "avg(Amount) as average_amount"
).show()

+----------------+
|  average_amount|
+----------------+
|88.3496192509521|
+----------------+



In [19]:
df_spark.groupBy("Class").agg(
    {"Amount": "avg"}
).show()

+-----+------------------+
|Class|       avg(Amount)|
+-----+------------------+
|    1|122.21132113821137|
|    0| 88.29102242233401|
+-----+------------------+



### Spark Processing

Apache Spark was used to load and preprocess the transaction dataset.
The data was converted into a Spark DataFrame, features were assembled
using VectorAssembler, standardized using StandardScaler, and divided
into training and testing datasets.

The Spark DataFrame contained 284,807 transactions and 30 input features.
The data was split into 227,790 training records and 57,017 testing records.